## Data Cleaning: Creating `sources` and `emissions_records` tables
The purpose of this script is to create two cleaned tables from the electricity-generated emissions and the emissions from non-residential onsite building usage based on data compiled from Climate TRACE. Two tables are created: `sources` (containing the source name and other attributes) and `emissions_records` (containing the emissions quantity for source records + other attributes). 

### Import Packages 

In [66]:
import pandas as pd

### Read in data
Read in the `power` and `non_res` datasets.

In [67]:
power = pd.read_csv('/Users/vedikashirtekar/MEDS/EDS-213/eds-213-labs/data/DATA/power/electricity-generation_emissions_sources_v5_5_0.csv')
non_res = pd.read_csv('/Users/vedikashirtekar/MEDS/EDS-213/eds-213-labs/data/DATA/buildings/non-residential-onsite-fuel-usage_emissions_sources_v5_5_0.csv')

It's always a good idea to explore the data types of each of the data frames.

In [68]:
# Explore the data structures
print(f"non_res shape: {non_res.shape}")
print(f"power shape: {power.shape}")

non_res shape: (207888, 43)
power shape: (154879, 43)


In [80]:
print(f"non_res dtypes:\n{non_res.dtypes}")

non_res dtypes:
source_id                          int64
source_name                       object
iso3_country                      object
start_time                datetime64[ns]
end_time                  datetime64[ns]
lat                              float64
lon                              float64
gas                               object
emissions_quantity               float64
temporal_granularity              object
activity                         float64
activity_units                    object
emissions_factor                 float64
emissions_factor_units            object
capacity                         float64
capacity_units                    object
capacity_factor                  float64
dtype: object


In [81]:
print(f"power dtypes:\n{power.dtypes}")

power dtypes:
source_id                          int64
source_name                       object
iso3_country                      object
start_time                datetime64[ns]
end_time                  datetime64[ns]
lat                              float64
lon                              float64
gas                               object
emissions_quantity               float64
temporal_granularity              object
activity                         float64
activity_units                    object
emissions_factor                 float64
emissions_factor_units            object
capacity                         float64
capacity_units                    object
capacity_factor                  float64
dtype: object


### Data Cleaning

For this analysis, we only need to extract the source- and emissions- related information. As such, we can drop the geospatial (except `lat` and `long`) and date modifications made to the data for each sector, as well as the "other" series of columns. 

In [ ]:
# Some columns (source_type, geometry_ref, sector, and subsector) are entirely null, not relevant, or have a constant value

# Store the drop columns in a list and drop them from both dataframes
drop_cols = ['source_type', 'geometry_ref', 'sector', 'subsector', 'created_date', 'modified_date']
non_res = non_res.drop(columns=drop_cols)
power = power.drop(columns=drop_cols)

In [ ]:
# Convert datetime columns
non_res['start_time'] = pd.to_datetime(non_res['start_time'])
non_res['end_time'] = pd.to_datetime(non_res['end_time'])

# Same for power
power['start_time'] = pd.to_datetime(power['start_time'])
power['end_time'] = pd.to_datetime(power['end_time'])

Check for any potential duplicate records for each year.

In [ ]:
# Are there any duplicates? 
# There should be one row per source_id + start_time + gas combination
duplicates = non_res.duplicated(subset=['source_id', 'start_time', 'gas'])
print(duplicates.sum())

0


Another critical step is to ensure that there are no NULL values in the numeric column that could affect the carbon intensity calculation.

There are some nulls recorded for geospatial information (`lat` and `long`); however, we can ignore these as location is not critical for our analysis. 

*Note: `lat` and `long` are included for potential future analysis for spatial comparisons.*

In [73]:
# Check for nulls in numeric fields
numeric_cols = ['emissions_quantity', 'activity', 'emissions_factor', 'capacity', 'lat', 'lon']
print(non_res[numeric_cols].isnull().sum())

emissions_quantity        0
activity                  0
emissions_factor          0
capacity                  0
lat                   15860
lon                   15860
dtype: int64


In [74]:
# Same for power
print(power[numeric_cols].isnull().sum())

emissions_quantity    0
activity              0
emissions_factor      0
capacity              0
lat                   0
lon                   0
dtype: int64


Both `power` and `non_res` contain several `other_` columns that should be dropped. We can use a `for loop` to loop through each dataframe and remove the columns starting with "other". 

In [75]:
# Drop "other" columns in non_res and power
other_cols = [c for c in non_res.columns if c.startswith('other')]
non_res = non_res.drop(columns=other_cols)

In [76]:
# Drop "other" columns in power
other_cols = [c for c in power.columns if c.startswith('other')]
power = power.drop(columns=other_cols)

### Extracting `sources` and `emissions_records` information
We can now extract the source and emissions quantity information for both sectors. The extracted `sources` and `emissions_records` tables can be exported as clean CSV files for simple database ingestion in SQL.

In [77]:
# Create sources table
sources_non_res = non_res[['source_id', 'source_name', 'iso3_country', 'start_time', 'end_time', 'lat', 'lon', 
                    'capacity', 'capacity_units', 'capacity_factor']].drop_duplicates(subset='source_id') # Keep only one row per source_id for sources table

sources_power = power[['source_id', 'source_name', 'iso3_country', 'start_time', 'lat', 'lon',
                        'capacity', 'capacity_units', 'capacity_factor']].drop_duplicates(subset='source_id') 

# Concatenate sources from non_res and power and keep only unique source_ids (no duplicates)
sources = pd.concat([sources_non_res, sources_power]).drop_duplicates(subset='source_id').reset_index(drop=True)

In [78]:
# Create emission_records table
record_cols = ['source_id', 'start_time', 'end_time', 'gas', 'emissions_quantity',
               'temporal_granularity', 'activity', 'activity_units', 
               'emissions_factor', 'emissions_factor_units']

# Add sector column to both dataframes and concatenate them
non_records = non_res[record_cols].copy()
non_records['sector'] = 'buildings'

power_records = power[record_cols].copy()
power_records['sector'] = 'power'

emission_records = pd.concat([non_records, power_records]).reset_index(drop=True)

In [79]:
# Export the cleaned tables to CSV for data ingestion
sources.to_csv('data/new_tables/sources.csv', index=False)
emission_records.to_csv('data/new_tables/emission_records.csv', index=False)